Welcome back to the lab again. From last week, you may have worked with mass spectrometry data and processed a bit of them by yourself. This lab will look into the results that you have made and we will practice a bit of data wrangling and visualization with Python. Thus, you can now understand the dataset much better with many more aspects from your result. As usual, please fill in your information here so we can give you some nice scores later. 


- Member1:
- Member2: 
- Contact email: 

Same here, there will be ten questions and three bonus questions for you to answer. Please try to connect this exercise to the lectures from the first weeks. The main goal of this lab is that you are not afraid to work with mass spectrometry as it is amazing. Woo hoo. 


## Intended learning outcomes (ILOs) 

On completion of the lab, the student should be able to:
```
* demonstrate data-processing procedures in mass spectrometry proteomics
* demonstrate the ability to answer statistical questions with computational tools in mass spectrometry
* identify quality of high-throughput dataset and handle with statistical understandings
* identify relevant issues in technologies and data with accessible visualization techniques
```
## Let's start! 

You may recall from what we have done in the first lab. Now, we want to look at them properly. Let's start with some basic Python programming skill. So, please just copy the file and change the path below. (If you don't remember, just take it from your submission)

In [ ]:
%config IPCompleter.greedy=True
%config InlineBackend.figure_format = 'retina'

In [ ]:
# Install if it's your first time to run
!pip install pandas numpy matplotlib seaborn missingno

In [ ]:
# Libraries ----
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno


# File path of the result
# Fix when export
path = 'lab2/sdrf_openms_design_msstats_in.csv'
ms_result = pd.read_csv(path)

# Data overview
Let's check how do your data look. You can do it basically with any spreadsheet or texteditor software. Python is also one of them so don't be afraid. 

In [ ]:
ms_result.head()
# or
# ms_result

## Q1. 
**What do you see in the output file? What are the columns and the rows** 

Ans.

You can now see that it's difficult to see them clearly as there are many rows. It's hard to get an overview of the dataset. Let's dig a bit deeper and make it a little more organized. Let's now check numbers of samples. These should be related to what you have with the SDRF file previously. 

In [ ]:
# Number of samples and what are they
(
    ms_result
    # Select column Reference and Condition
    [['Reference', 'Condition']]
    # Remove duplicated rows in the table
    .drop_duplicates()
)

Does it look similar? Definitely, haha. 

Now, look at the proteins. 

## Q2. 
**Let's check the peptides and proteins. How many unique proteins and peptides? What percentage of matched protein from your library?**

In [ ]:
# Peptides and Protein 
# Extract a table of three columns with unique proteins, peptides and precursor charge

# ms_result[[_, _, _]] \
#     .drop_duplicates()

Ans.

## Q3. 
**Do you see any peptide modification during sample preparation? If yes, why do we need them?**

Ans. 


## BQ1. 
**Please show a summarised table containing numbers of protein counts and the detectable peptide numbers. For example, there are 10 proteins and each of them has 5 detectable peptides.**

In [ ]:
# Peptide per protein count
# Hint: value_counts() or groupby().size()
# This will give you numbers of duplicated items in a particular column

# Dynamic range

Now, let's roughly look at the intensity of the peptides. We will use Python (matplotlib and seaborn) to visualize the data.

In [ ]:
# Concatenate PeptideSequence and PrecursorCharge to new column
ms_result['PeptideSequencePC'] = (
    ms_result['PeptideSequence'] + '_' + ms_result['PrecursorCharge'].astype(str)
)

# Order peptide/precursors along the x-axis by their (mean) intensity
order = (
    ms_result.groupby('PeptideSequencePC')['Intensity']
    .mean()
    .sort_values()
    .index
)
x_pos = {name: i for i, name in enumerate(order)}
plot_df = ms_result.assign(x_pos=ms_result['PeptideSequencePC'].map(x_pos))

fig, ax = plt.subplots(figsize=(10, 6))

# plot all peptide/precursors, colored by Reference
sns.scatterplot(
    data=plot_df, x='x_pos', y='Intensity', hue='Reference', alpha=0.4, ax=ax
)

# rolling-mean trend line per Reference, as an approximation of geom_smooth()
for ref, group in plot_df.groupby('Reference'):
    group = group.sort_values('x_pos')
    trend = group['Intensity'].rolling(window=50, min_periods=1, center=True).mean()
    ax.plot(group['x_pos'], trend)

ax.set_yscale('log')
ax.set_xticks([])
ax.set_xlabel('Peptides')
ax.set_ylabel('Intensity')
ax.set_title('Overall peptide Intensity')
ax.legend(title='Reference', loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=3)

plt.tight_layout()
plt.show()

You now can see the dynamic range of every peptides that were detected. The intensity of the peptides is quite different maybe in some samples.

## Q4. 
**What can we imply from this plot? What is the dynamic range of the dataset?**

Ans.

## Q5. 
**Pick one protein that is comprised of 10 detectable peptides. Visualize the peptide intensity. Do we see every peptide in every sample? Does every peptide have the same intensity in each sample for each peptide. If not, why?**

In [ ]:
# Select a protein with 10 peptides that are detected in every sample

# Print out the protein name


# Visualize it

Ans.

# Missing data 
Let's now visualize the data that you already have with their intensities. We will look the data at the peptide precursor level. 

In [ ]:
select_pept = (
    ms_result[['Reference', 'PeptideSequence', 'PrecursorCharge', 'Intensity']]
    .pivot_table(index=['PeptideSequence', 'PrecursorCharge'], columns='Reference', values='Intensity')
)

msno.matrix(select_pept)
plt.show()

The problem now is that we can detect some missing data in the dataset. This is a common problem in mass spectrometry data. We can see that some peptides are not detected in some samples. 


## Q6. 
**Why there are missing values with MS?**


Ans. 


Let's now select good quality peptide based on the missing data. We will remove the peptides that are not detected in more than 50% of the samples.

## Q7. 
**From `select_pept` table, remove the peptides that are not detected in more than 50% of the samples. How many unique proteins and peptides are left?**

In [ ]:
# Remove the peptides that are not detected in more than 50% of the samples
# Hint: melt() and groupby()/filter()


# Select filtered peptides 
ms_result_filt = ms_result.merge(filter_pept, on=['PeptideSequence', 'PrecursorCharge'])


# Protein count


# Peptide count

Looks like we are more confident with the data now. Let's now calculate the protein abundance. The rule is we expect all peptide precursors to be detected with similar intensity in every protein. Meaning that, we can average the intensity of the peptides in each protein and compare them with the rest.

## Q8.
**What are the protein abundances in each sample?**

In [ ]:
# Hint: groupby() and .mean()

# Protein concentration
## Q9.
**Let's plot a dynamic range of protein concentration in each sample. What can we imply from this plot?**

In [ ]:
# Plot

## Q10. 
**Visualize the concentration of protein from PSMB3 gene with boxplot. What can we summarise here? Additionally, if it is in your experiment, how would you proceed?**

In [ ]:
# hint
# Fix this code below

# Select Protein
prot = (
    prot_level
    .merge(ms_result[['Reference', 'Condition']].drop_duplicates(), on='Reference')
    [[__, _, __]]
    .query("ProteinName == '____'")
)


# # Boxplot

## BQ2. 
**Plot the abundance of the most differentiated protein**

In [ ]:
# # Boxplot

## BQ3.
**What are the advantages and disadvantages of MS Proteomics?**